# Background

Author: Diane Menuz  
Date: February 2, 2026   
  
Goal: Compile data for a single site and run through the preprocess function.
  
During the preprocess stage, identify data that is missing within a data type (met or eddy) and gap-  
fill if possible.

**Requirements**
- raw_fold: folder that has the following
- within raw_fold, subfolders with each stationid
- within station-specific folder, the following files:
    - {stationname}_Flux_AmeriFluxFormat.dat: Ameriflux file downloaded from EasyFlux; should only  
    be one
    - {stationname}_Flux_CSFormat.dat: CSFLUX file downloaded from EasyFlux; should only be one
    - folder called AmeriFluxFormat with eddy data downloaded from the station
    - folder called Statistics_Ameriflux with met data downloaded from the station
    - folder called Statistics with met data downloaded from the station; needs to be converted with  
    card convert
    - folder called Flux_CSFormat with eddy data downloaded from the station; needs to be converted  
    with card convert


# Initialization

## Parameters

In [ ]:
# interval of data you want to work with: 30 or 60
# subset of data chosen based on the interval_updates.py dictionary
interval = 30
# target site you will be running
stationid = 'US-UTD'

micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"

Below are defaults that you will not need to change unless changing file structure

In [ ]:
# folder with items outlined above
raw_fold_path = 'Shared drives/UGS_Flux/Data_Downloads/compiled'
output_folder = 'Shared drives/UGS_Flux/Data_Downloads/compiled/preprocessed_site_data'

#amflux column data
amflux_dat = r'Shared drives/UGS_Flux/Data_Processing/site_specific_data_review/flux-met_processing_variables_20250818.csv'

In [ ]:
# dictionary of statoinid and folder names
# will subset out the site we are interested in using target_site from above
site_folders = {'US-UTD':'Dugout_Ranch',
                 'US-UTB':'BSF',
                 'US-UTJ':'Bluff',
                 'US-UTW':'Wellington',
                'US-UTE':'Escalante',
                 'US-UTM':'Matheson',
                 'US-UTP':'Phrag',
                'US-CdM':'Cedar_mesa',
                 'US-UTV':'Desert_View_Myton',
                 'US-UTN':'Juab',
                'US-UTG':'Green_River',
                 'US-UTL':'Pelican_Lake',
                 }

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import sys
import pathlib
from pathlib import Path


import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px

import pandas as pd
import numpy as np
from pandas.tseries.frequencies import to_offset
import plotly.graph_objects as go

sys.path.append(micromet_path)
import micromet
from micromet import validate
from micromet import interval_updates
from micromet import file_compile
from micromet import eddy_plots as ed_plot

%matplotlib inline

## Initialize Logger

In [ ]:
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setFormatter(
    logging.Formatter(
        fmt="%(levelname)s [%(asctime)s] %(name)s – %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
logger.addHandler(ch)

## Set Path

In [ ]:
raw_fold = pathlib.Path(raw_fold_path)
amflux = pd.read_csv(amflux_dat)

## Functions

In [ ]:
def preprocess_data(stationid,
                    parent_fold,
                    glob_name,
                    interval,
                    skip_rows):
    """
    Reads, preprocesses, and concatenates environmental data files (MET or Eddy 
    Covariance) from a directory based on a glob pattern.

    The function dynamically determines the data type ('met' or 'eddy') based 
    on the file name pattern and applies specific cleaning steps, including 
    column name truncation and datetime conversion, before passing data to 
    the micromet.Reformatter.

    Parameters
    ----------
    stationid : str
        Unique identifier for the station (e.g., 'US-ABC').
    parent_fold : pathlib.Path or str
        The root directory object to search for files within.
    glob_name : str
        The glob pattern used to find files (e.g., 'TOA5*Statistics*.dat').
        The pattern determines the internal data type ('met' or 'eddy').
    interval : str or int
        The measurement interval for the data (e.g., '30min'). Used by 
        the external Reformatter.
    skip_rows : bool
        If True, skips the first few metadata rows (0, 2, 3) and applies 
        specific NaN handling and TIMESTAMP conversion.

    Returns
    -------
    pandas.DataFrame or None
        A single, concatenated and cleaned DataFrame indexed by ('STATIONID', 
        'DATETIME_END') with duplicate indices removed, or None if no files 
        matching the glob pattern are found.

    Notes
    -----
    1. The function assumes the presence of external objects 'micromet' and 'logger'.
    2. Column names ending in '_Avg' or '_Tot' are truncated (e.g., 'Ta_Avg' -> 'Ta').
    3. If multiple files yield the same 'STATIONID'/'DATETIME_END' index, only 
       the first record is kept (`keep='first'`).
    """
    output_dict = {}
    if glob_name in ['TOA5*Statistics*.dat','*Statistics_AmeriFlux*.dat']:
        data_type = 'met'
        drop_soil = False
    elif glob_name in ['*_Flux_AmeriFluxFormat.dat', "*_Flux_CSFormat*.dat",
             "*Flux_AmeriFluxFormat*.dat"]:
        data_type = 'eddy'
        drop_soil = True
    else: 
        print(f'Glob name {glob_name} not a reconized ame')
    for file_name in parent_fold.glob(f"{glob_name}"):
        print(f"Processing file: {file_name}")
        if skip_rows == True:
            sts = pd.read_csv(file_name, skiprows = [0,2,3],
                             na_values=[-9999,"NAN","NaN","nan"])
            sts['TIMESTAMP'] = pd.to_datetime(sts['TIMESTAMP'])
            sts["TIMESTAMP_END"] = sts.TIMESTAMP.dt.strftime(
                "%Y%m%d%H%M").astype(int)
        else:
            sts = pd.read_csv(file_name, encoding_errors='ignore')
        for col in sts.columns:
            if col.endswith("_Avg"):
                sts.rename(columns={col: col[:-4]}, inplace=True)
            elif col.endswith("_Tot"):
                sts.rename(columns={col: col[:-4]}, inplace=True)
        am_data = micromet.Reformatter(drop_soil=drop_soil, logger=logger,)
        df = am_data.preprocess(sts, data_type=data_type, interval=interval)
        output_dict[file_name.stem] = df
    if not output_dict: # This checks if the dictionary is empty (contains no items)
        print(f"No files found for {stationid}")
        output = None 
    else:
        output_concat = pd.concat(output_dict)
        print("Files successfully concatenated using dictionary keys.")
        output_concat = pd.concat(output_dict)
        output = output_concat.reset_index(level=0)
        output['STATIONID'] = stationid
        output = output.reset_index().rename(columns={'level_0':'FILE_NAME'})
        output = output.set_index(['STATIONID','DATETIME_END'])
        output = output[~output.index.duplicated(keep='first')]
    return(output)

# Compiling Files

**Do not need to rerun this each time MicroMet is changed**  
Just rerun for sites that need their data compiled. When most of the data are compiled, it is  
sometimes just easier to copy over a file or two that was missed. 

This process is not converting TOA files for now. Best I have come up with is moving the raw files  
to a Statistics_Raw folder, manually converting with Card Convert, and then compiling to the final  
folder in a second step. Seems like some files are getting missed, though

In [ ]:
# data types to compile. 
# key is the term within the file name to search for
# value is the folder name to copy the files to
# note that the statistics name is a regex expressions where \d means that name 
# is followed by one or more digits

folder_search_term = {
    'Statistics_Ameriflux':"Statistics_AmeriFlux",
    r"Statistics_\d+":"Statistics_Raw",
    'Operatn_Notes':'System_Operatn_Notes',
    'Config_Setting_Notes':'Config_Setting_Notes',
    'Flux_AmeriFluxFormat':'AmeriFluxFormat',
    'Flux_CSFormat':'Flux_CSFormat',
    'Flux_Notes':'Flux_Notes'
}

In [ ]:
for file, folder in folder_search_term.items():
    print(f'Compiling data into {folder_search_term[file]} folder')
    contains = file
    raw_folder = Path(f'M:/Shared drives/UGS_Flux/Data_Downloads/{site_folders[stationid]}')
    outdir = Path(f'M:\\Shared drives\\UGS_Flux\\Data_Downloads\\compiled\\{stationid}\\{folder_search_term[file]}')
    file_compile.compile_files(raw_folder,outdir, contains)

In [ ]:
# after running the above, Card Convert files in the Statistics_Raw folder and then run the following
# next compilation for Statistics_Converted to Statistics folder (after converting the raw files

contains = 'TOA5'
raw_folder = Path(
        f'M:/Shared drives/UGS_Flux/Data_Downloads/compiled/{stationid}/Statistics_Converted')
outdir = Path(
        f'M:\\Shared drives\\UGS_Flux\\Data_Downloads\\compiled\\{stationid}\\Statistics')
file_compile.compile_files(raw_folder,outdir, contains)

In [ ]:
# optional to card convert and compile CSFlux data
# usually you can get the csflux data from the easyflux download, but that may not be the case
# everywhere, such as dugout

# will need to createa a Flux_CSFormat_Converted folder with the converted files

contains = 'TOA5'
raw_folder = Path(
        f'M:/Shared drives/UGS_Flux/Data_Downloads/compiled/{stationid}/Flux_CSFormat_Converted')
outdir = Path(
        f'M:\\Shared drives\\UGS_Flux\\Data_Downloads\\compiled\\{stationid}\\Flux_CSFormat')
file_compile.compile_files(raw_folder,outdir, contains)

# Met

## Compile Met Statistics Tables

In [ ]:
parent_folder = Path(os.path.join(raw_fold, stationid, 'Statistics'))
met_stats = preprocess_data(stationid,
                parent_folder,
                glob_name="TOA5*Statistics*.dat",
                interval=interval,
                skip_rows=True)

In [ ]:
# check variable naming, skipping some we already know are not in ameriflux
substrings = ['EC_', 'K_', 'LWMCON_', 'LWMDRY_', "LWMV_",
              'FILE_NAME', 'RECORD', 'SAMPLING_INTERVAL',
              'T_NR_OUT', 'SN500_HEATER_SECS', 'T_NR',
              'T_SI111_BODY', 'V_BATT_MET', 'WND_DIR_SD1_WVT',
              'T_PANEL']
pattern = '|'.join(substrings)
mask = met_stats.columns.str.contains(pattern, regex=True)
columns_to_drop = met_stats.columns[mask]
stats_met_drop_soilvue_cols = met_stats.drop(columns=columns_to_drop)
results = validate.compare_names_to_ameriflux(
    stats_met_drop_soilvue_cols, amflux)

In [ ]:
# subset data by the interval (30 or 60)
stats_met_interval = interval_updates.subset_interval(
    met_stats,interval_updates.interval_update_dict, 
    interval, data_type='met' )

In [ ]:
# plot data to examine missing data
plot_data = stats_met_interval.loc[stationid].sort_index()
ed_plot.plotlystuff([plot_data, plot_data], ['WD_1_1_2','NETRAD_1_1_2'], 
            chrttitle=f'{stationid}')

In [ ]:
# review columns
for col in met_stats.columns.sort_values():
    print(col)

In [ ]:
# export data
stats_met_interval.to_parquet(f'{output_folder}/{stationid}_{interval}_metstats_preprocessed.parquet')

## Compile Statistics Ameriflux .dat Tables

In [ ]:
parent_folder = Path(os.path.join(raw_fold, stationid, 'Statistics_Ameriflux'))
afstats_met_temp = preprocess_data(stationid,
                parent_folder,
                glob_name="*Statistics_AmeriFlux*.dat",
                interval=interval,
                skip_rows=False)

In [ ]:
# check on naming issues with leaf wetness
# should be 1_2_1 not 1_1_2
# and should have LEAF_WET and not LWMWET
# may sometimes be duplicated leaf wetness values
substrings = ['LEAF', 'LWM']
pattern = '|'.join(substrings)
mask = afstats_met_temp.columns.str.contains(pattern, regex=True)
afstats_met_temp.columns[mask].sort_values()

In [ ]:
# fixing leaf wetness columns if there are any issues

afstats_met = afstats_met_temp.copy()

# # rename any bad names
rename_dict = {
        "LWMCON_1_1_2": 'LWMCON_1_2_1',
        "LWMDRY_1_1_2": 'LWMDRY_1_2_1',
        "LWMV_1_1_2": 'LWMV_1_2_1',
        "LWMWET_1_1_2": 'LEAF_WET_1_2_1',
        "LWMWET_1_1_1": 'LEAF_WET_1_1_1'
    }

afstats_met = afstats_met.rename(columns=rename_dict)

In [ ]:
# # Identify columns with all NA values and drop b/c lots of weird columns
all_na_series = afstats_met.isna().all()
columns_to_drop = all_na_series[all_na_series].index.tolist()
print(columns_to_drop)
afstats_met.drop(columns=columns_to_drop, inplace=True)

In [ ]:
# review columns
for col in afstats_met.columns.sort_values():
    print(col)

In [ ]:
# review fields that show up- anything off base?
# not reviewing soilvue and leaf wetness variables
substrings = ['EC_', 'K_', 'LWMCON_', 'LWMDRY_', "LWMV_"]
pattern = '|'.join(substrings)
mask = afstats_met.columns.str.contains(pattern, regex=True)
columns_to_drop = afstats_met.columns[mask]
temp = afstats_met.drop(columns=columns_to_drop)
results = validate.compare_names_to_ameriflux(temp, amflux)

In [ ]:
afstats_met_interval = interval_updates.subset_interval(
    afstats_met,interval_updates.interval_update_dict, 
    interval, data_type='met' )

In [ ]:
# plot data to examine missing data
plot_data = afstats_met_interval.loc[stationid].sort_index()
ed_plot.plotlystuff([plot_data, plot_data], ['WD_1_1_2','NETRAD_1_1_2'], 
            chrttitle=f'{stationid}')

In [ ]:
afstats_met_interval.columns

In [ ]:
# export data
afstats_met_interval.to_parquet(f'{output_folder}/{stationid}_{interval}_metstatsaf_preprocessed.parquet')

## Look For Met Record Gaps

Identify data missing in both met records and fill in gaps.

In [ ]:
afstats_plot = afstats_met_interval.loc[stationid].sort_index()
metstats_plot = stats_met_interval.loc[stationid].sort_index()

ed_plot.plotlystuff([afstats_plot, metstats_plot], ['WD_1_1_2','WD_1_1_2'], 
            chrttitle=f'{stationid}, Green is Ameriflux Stats and Pink is Statistics')

# Compile Eddy from Easyflux Web

# Compile Eddy from Datalogger

## Compile Ameriflux Format dat files from Dataloggers

In [ ]:
parent_folder = Path(os.path.join(raw_fold, stationid, "AmeriFluxFormat"))

eddyaf_dl = preprocess_data(stationid, parent_folder,
                glob_name = "*Flux_AmeriFluxFormat*.dat",
                interval = interval,
                skip_rows=False)

In [ ]:
# Identify columns with all NA values and drop b/c lots of weird columns
all_na_series = eddyaf_dl.isna().all()
columns_to_drop = all_na_series[all_na_series].index.tolist()
print(columns_to_drop)
eddyaf_dl.drop(columns=columns_to_drop, inplace=True)

In [ ]:
# data validation
results = validate.compare_names_to_ameriflux(eddyaf_dl, amflux)
print('\n')

temp = eddyaf_dl.reset_index(level=1)
validate.validate_timestamp_consistency(temp)

In [ ]:
for col in eddyaf_dl.columns.sort_values():
    print(col)

In [ ]:
# subset out 30 minute data
eddyaf_dl_interval = interval_updates.subset_interval(
    eddyaf_dl,interval_updates.interval_update_dict, 
    interval, data_type='eddy' )

In [ ]:
# plot data to examine missing data
plot_data = eddyaf_dl_interval.loc[stationid].sort_index()
ed_plot.plotlystuff([plot_data, plot_data], ['WD_1_1_1','TA_1_4_1'], 
            chrttitle=f'{stationid}')

In [ ]:
# export data
eddyaf_dl_interval.to_parquet(f'{output_folder}/{stationid}_{interval}_eddyaf_dl_preprocessed.parquet')

## Compile CSFormat Files from Raw Data

In [ ]:
parent_folder = Path(os.path.join(raw_fold, stationid, "Flux_CSFormat"))

eddycs_dl = preprocess_data(stationid, parent_folder,
                glob_name = "*_Flux_CSFormat*.dat",
                interval = interval,
                skip_rows=True)

In [ ]:
 #dropping/renaming variables
drop_fields = [
    "TS_CS65X_2_1_1",
    "WS_RSLT",
    "_229_DEL_TMPR(1)",
    "_229_DEL_TMPR(2)",
    "_229_TMPR_T0_1",
    "_229_TMPR_T0_2",
    "_229_TMPR_T1_1",
    "_229_TMPR_T1_2",
    "_229_TMPR_T30_1",
    "_229_TMPR_T30_2",
    "_PANEL_TMPR_T0",
    "_PANEL_TMPR_T1",
    "_PANEL_TMPR_T30",
    "WND_DIR_STD",
    "WND_DIR_UNIT_VEC",
    "WND_SPD_AVG",
    "U_HEATMAX",
    "U_SEN0",
    "U_SENAMP",
    "U_SENMAX",
    "SONIC_AZIMUTH",
    "CS65X_EC_2_1_1"
    "SUN_AZIMUTH",
    "SUN_DECLINATION",
    "SUN_ELEVATION",
    "HEIGHT_AGL",
    "HOUR_ANGLE",
    "CS65X_PERM_1_1_1",
    "DAYTIME",
    "E",
    "E1_Q",
    "ANONYMOUS1",
    "ANONYMOUS2",
    "TD_TP01",
    "AIR_MASS_COEFF",
    "ROCP_TP01",
    "Q"
]

for field in drop_fields:
    if field in eddycs_dl.columns:
        print(f'Dropping {field}')
        eddycs_dl = eddycs_dl.drop(columns=[field],axis=1)

rename_fields = {
    "CS65X_EC_1_1_1":"EC_1_1_1",
    "CS65X_EC_1_1_2":"EC_1_1_2",
    "LI7700_AMB_TMPR":"TA_1_1_5",
    "T_SONIC":"T_SONIC_1_1_1",
    'CO2_SIGMA':'CO2_SIGMA_1_1_1', 
    'H2O_SIGMA':'H2O_SIGMA_1_1_1',
    }


eddycs_dl = eddycs_dl.rename(columns=rename_fields)

In [ ]:
# data validation
results = validate.compare_names_to_ameriflux(eddycs_dl, amflux)
print('\n')

temp = eddycs_dl.reset_index(level=1)
validate.validate_timestamp_consistency(temp)

In [ ]:
# subset out 30 minute data
eddycs_dl_interval = interval_updates.subset_interval(
    eddycs_dl,interval_updates.interval_update_dict, 
    interval, data_type='eddy' )

In [ ]:
# plot data to examine missing data
plot_data = eddycs_dl_interval.loc[stationid].sort_index()
ed_plot.plotlystuff([plot_data, plot_data], ['WD_1_1_1','TA_1_4_1'], 
            chrttitle=f'{stationid}')

In [ ]:
# export data
eddycs_dl_interval.to_parquet(f'{output_folder}/{stationid}_{interval}_eddycsflux_dl_preprocessed.parquet')

## Compare missing datalogger data

In [ ]:
ameriflux = eddyaf_dl_interval.loc[stationid].sort_index()
csflux = eddycs_dl_interval.loc[stationid].sort_index()

ed_plot.plotlystuff([ameriflux, csflux], ['WD_1_1_1','WD_1_1_1'], 
            chrttitle=f'{stationid}, Green is Ameriflux and Pink is CSFlux')